## Importação de Pacotes

In [51]:
import pandas as pd
import re

## Contato Inicial

### Informações sobre as Características
titulo - nome do livro
codigo_titulo - codigo de identificação no sistema da biblioteca
codigo_conhecimento - área do conhecimento
autor - nome(s) do(s) autor(es)
curso - curso o qual aquele exemplar se destina
campus - locaização da biblioteca
data_divulgacao - provavelmente ano de aquisição
quantidade_exemplares - número de exemplares

### Carregamento dosDados
Ao tentar carregar os dados em um dataframe no pandas, houve algun problemas relacionados a organização dos dados:
1. Os separadores de coluna não usam vírgula, nem ponto e vírgula mas \t (tab)
2. A codificação de caractere é legado (latin1), não é o padrão de hoje (utf-8), algo comum em órgãos públicos brasileiros.
3. Títulos de livros podem conter aspas duplas, mas ao tentar carregar, o pandas entende como separador de colunas para que se possa adicionar caracteres especiais como vírgula, que seriam entendidos também como separadores de coluna. Nesse caso se o título tiver aspas, usando bom senso é por que faz parte do dado, então fui buscar como se ignora isso, basta fazer quoting=3, que diz que é para ignorar as aspas, tratá-las como texto normal.
4. Teve um erro 'Expected 9 fields in line 2502, saw 10', ou seja, a maioria dos dados possuem 9 colunas mas havia 10 naquela linha, provavelmente um humano errou na hora de montar esses dados e criou uma sem querer, quem sabe, de qualquer forma geralmente é um erro então pedimos para que o pandas ignore a décima coluna na hora de montar o dataframe.

In [52]:
df = pd.read_csv('Dados/ifce-biblioteca-livros.csv', sep='\t', encoding='utf-8-sig', quoting=3, on_bad_lines='skip')
print(df.head())

                                      "titulo"  "codigo_titulo"  \
0                       A arte da criatividade         140771.0   
1                                       !Vale!          33058.0   
2  (A) Teoria da relatividade especial e geral         107280.0   
3                              [Uirá dos reis]          84019.0   
4                        "10 anos com Mafalda"          81914.0   

  "codigo_conhecimento"              "autor"  \
0                   NaN         Judkins, Rod   
1                     .  Alves, Adda-Nari M.   
2                   NaN     Einstein, Albert   
3                   NaN                  NaN   
4                     .                Quino   

                           "publicacao"  \
0         Rio de Janeiro : Rocco , 2015   
1            São Paulo : Moderna , 1997   
2   Rio de Janeiro : Contraponto , 1999   
3  Fortaleza : Expressão Gráfica , 2009   
4     São Paulo : Martins Fontes , 2011   

                                             "cur

In [53]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 73472 entries, 0 to 73471
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   "titulo"                 73472 non-null  str    
 1   "codigo_titulo"          73471 non-null  float64
 2   "codigo_conhecimento"    40839 non-null  str    
 3   "autor"                  60436 non-null  str    
 4   "publicacao"             73471 non-null  str    
 5   "curso"                  23668 non-null  str    
 6   "campus"                 72691 non-null  str    
 7   "data_divulgacao"        28325 non-null  str    
 8   "quantidade_exemplares"  73471 non-null  float64
dtypes: float64(2), str(7)
memory usage: 5.0 MB
None


In [54]:
print(df.describe())

       "codigo_titulo"  "quantidade_exemplares"
count     73471.000000             73471.000000
mean      59725.064257                 3.878687
std       38924.630945                 5.043817
min           1.000000                 0.000000
25%       26171.500000                 1.000000
50%       51320.000000                 2.000000
75%       86550.500000                 5.000000
max      143149.000000                76.000000


Pode-se notar que há muitas colunas vazias, com o tipo de dado incorreto e ouras que iremos querer remover, modificar ou criar.

In [55]:
print(df.isnull().sum())

"titulo"                       0
"codigo_titulo"                1
"codigo_conhecimento"      32633
"autor"                    13036
"publicacao"                   1
"curso"                    49804
"campus"                     781
"data_divulgacao"          45147
"quantidade_exemplares"        1
dtype: int64


In [56]:
print(df.columns.tolist())

['"titulo"', '"codigo_titulo"', '"codigo_conhecimento"', '"autor"', '"publicacao"', '"curso"', '"campus"', '"data_divulgacao"', '"quantidade_exemplares"']


In [57]:
df_drop = df.drop(['"codigo_titulo"', '"codigo_conhecimento"', '"curso"', '"data_divulgacao"'], axis=1)

In [58]:
print(df_drop.head())

                                      "titulo"              "autor"  \
0                       A arte da criatividade         Judkins, Rod   
1                                       !Vale!  Alves, Adda-Nari M.   
2  (A) Teoria da relatividade especial e geral     Einstein, Albert   
3                              [Uirá dos reis]                  NaN   
4                        "10 anos com Mafalda"                Quino   

                           "publicacao"                   "campus"  \
0         Rio de Janeiro : Rocco , 2015             CAMPUS ARACATI   
1            São Paulo : Moderna , 1997              CAMPUS IGUATU   
2   Rio de Janeiro : Contraponto , 1999  CAMPUS TABULEIRO DO NORTE   
3  Fortaleza : Expressão Gráfica , 2009             CAMPUS ARACATI   
4     São Paulo : Martins Fontes , 2011               CAMPUS CRATO   

   "quantidade_exemplares"  
0                      5.0  
1                      1.0  
2                      1.0  
3                      1.0  
4      

In [59]:
print(df_drop.isnull().sum())

"titulo"                       0
"autor"                    13036
"publicacao"                   1
"campus"                     781
"quantidade_exemplares"        1
dtype: int64


In [60]:
df_drop['"quantidade_exemplares"']=df_drop['"quantidade_exemplares"'].fillna(0).astype(int)

In [61]:
df_drop.columns = ["titulo", "autor", "pub", "campus", "exemplares"]

In [62]:
df_drop["campus"] = df_drop["campus"].str.replace("campus ", "", case=False, regex=False)

In [63]:
print(df_drop.head())

                                        titulo                autor  \
0                       A arte da criatividade         Judkins, Rod   
1                                       !Vale!  Alves, Adda-Nari M.   
2  (A) Teoria da relatividade especial e geral     Einstein, Albert   
3                              [Uirá dos reis]                  NaN   
4                        "10 anos com Mafalda"                Quino   

                                    pub              campus  exemplares  
0         Rio de Janeiro : Rocco , 2015             ARACATI           5  
1            São Paulo : Moderna , 1997              IGUATU           1  
2   Rio de Janeiro : Contraponto , 1999  TABULEIRO DO NORTE           1  
3  Fortaleza : Expressão Gráfica , 2009             ARACATI           1  
4     São Paulo : Martins Fontes , 2011               CRATO           1  


In [64]:
print(df_drop['campus'].value_counts())

campus
FORTALEZA                            11145
IGUATU                                6808
CRATO                                 6028
LIMOEIRO DO NORTE                     4846
CEDRO                                 3752
MARACANAÚ                             3615
JUAZEIRO DO NORTE                     3130
ARACATI                               2969
CANINDÉ                               2603
SOBRAL                                2548
CRATEÚS                               2232
QUIXADÁ                               2219
ACARAÚ                                1735
CAUCAIA                               1712
TIANGUÁ                               1509
TAUÁ                                  1471
BATURITÉ                              1378
TABULEIRO DO NORTE                    1323
UBAJARA                               1283
ITAPIPOCA                             1191
MORADA NOVA                           1112
LIMOEIRO DO NORTE ( CIDADE ALTA )     1063
JAGUARIBE                              982
BOA 

In [65]:
split_result = df_drop['pub'].str.split(' : ', n=1, expand=True)

split_result = split_result.reindex(columns=[0, 1])

df_drop[['origem-livro', 'editora-ano']] = split_result

split_result = df_drop['editora-ano'].str.split(' , ', n=1, expand=True)

split_result = split_result.reindex(columns=[0, 1])

df_drop[['editora', 'ano']] = split_result

In [68]:
df_clean = df_drop.drop(['pub', 'editora-ano'], axis=1)

In [70]:
print(df_clean.head())

                                        titulo                autor  \
0                       A arte da criatividade         Judkins, Rod   
1                                       !Vale!  Alves, Adda-Nari M.   
2  (A) Teoria da relatividade especial e geral     Einstein, Albert   
3                              [Uirá dos reis]                  NaN   
4                        "10 anos com Mafalda"                Quino   

               campus  exemplares    origem-livro            editora   ano  
0             ARACATI           5  Rio de Janeiro              Rocco  2015  
1              IGUATU           1       São Paulo            Moderna  1997  
2  TABULEIRO DO NORTE           1  Rio de Janeiro        Contraponto  1999  
3             ARACATI           1       Fortaleza  Expressão Gráfica  2009  
4               CRATO           1       São Paulo     Martins Fontes  2011  


In [71]:
caracteres = set(''.join(df_clean['titulo'].dropna()).translate(
    str.maketrans('', '', 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 àáâãéêíóôõúüçÀÁÂÃÉÊÍÓÔÕÚÜÇ')
))
print(caracteres)

{'%', '°', '+', '-', 'º', 'Ò', '"', 'ñ', '=', ';', '³', 'î', '(', ':', ']', '$', ')', 'ì', 'ö', '&', 'Ì', ',', '<', 'ª', 'è', '®', '>', '[', '²', '#', '\xa0', '_', 'ë', '¿', '@', '¡', '*', '/', '!', '.', '…', "'", '?'}


In [72]:
df_clean['titulo'] = df_clean['titulo'].str.replace(
    r'[""\'\'()\[\]{}\xa0…ºª®¿¡#@*_=;<>$²³°/\\]', '', regex=True
)

df_clean['titulo'] = df_clean['titulo'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [74]:
df_clean['titulo'].head()

0                       A arte da criatividade
1                                       !Vale!
2    A Teoria da relatividade especial e geral
3                                Uirá dos reis
4                          10 anos com Mafalda
Name: titulo, dtype: str

In [76]:
df_classif = df_clean[['titulo']].copy()

df_classif['titulo'] = df_classif['titulo'].str.lower()

df_classif['titulo'] = df_classif['titulo'].str.replace(r'[^\w\s]', '', regex=True)

df_classif['titulo'] = df_classif['titulo'].str.strip()

df_classif['titulo'] = df_classif['titulo'].str.replace(r'\s+', ' ', regex=True)

In [77]:
df_classif.head()

,titulo
0,a arte da criatividade
1,vale
2,a teoria da relatividade especial e geral
3,uirá dos reis
4,10 anos com mafalda


ainda vou definir um modelo aqui...

In [ ]:
areas_alvo = ["Ciências Exatas e da Terra", "Engenharias", "Ciências Humanas e Sociais", "Ciências Biológicas e da Saúde", "Linguística e Artes", "Literatura"]

resultados_classificacao = []

for texto in df_classif['titulo']:

    area_escolhida = modelo.predict(texto, areas_alvo)

    resultados_classificacao.append(area_escolhida)

df_classif['area'] = resultados_classificacao